<a href="https://colab.research.google.com/github/ancientphoenix34/Fine-Tuning/blob/main/FineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Installing necessary libraries for fine-tuning and evaluation...")

!pip install -q transformers accelerate bitsandbytes torch datasets peft trl openai scikit-learn gradio

print("Libraries installed successfully!")

In [ ]:
import os
from huggingface_hub import login, notebook_login
print("Attempting Hugging Face login...")

notebook_login()
print("HF Login successful (or token already present)!")

In [ ]:
import torch
import transformers
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,  # For quantization config
)
from datasets import load_dataset, Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training  # PEFT tools
from trl import SFTTrainer  # The fine-tuning trainer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import json  # For OpenAI JSONL format
import time  # For waiting on OpenAI jobs
from IPython.display import display, Markdown
import random

print("Core libraries imported.")

# Check for GPU availability (Crucial for fine-tuning)
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected. Fine-tuning requires a GPU. Please enable GPU in Runtime settings.")
    # Optionally exit or raise an error here


In [ ]:
# Helper function for markdown display
def print_markdown(text):
    """Displays text as Markdown."""
    display(Markdown(text))

In [ ]:
# Link to the dataset on Hugging Face: https://huggingface.co/datasets/Daniel-ML/sentiment-analysis-for-financial-news-v2/viewer
dataset_id = "Daniel-ML/sentiment-analysis-for-financial-news-v2"

print(f"Loading dataset: {dataset_id}...")
labeled_dataset = load_dataset(dataset_id, split = "train")  # Load the main split
print("Dataset loaded successfully!")

# Let's view the dataset
print("\n--- Dataset Information ---")
print(labeled_dataset)

In [ ]:
# Let's view the dataset features
print("\n--- Dataset Features ---")
print(labeled_dataset.features)

In [ ]:
labels = labeled_dataset.to_pandas()['sentiment'].unique().tolist()
print(f"Unique labels in the dataset: {labels}")

In [ ]:
# Let's view the data as a Pandas DataFrame
display(labeled_dataset.select(range(5)).to_pandas()[["text", "sentiment"]])

In [ ]:
# Let's Split the Data into Train and Test Sets ---
print("\nSplitting data into Train (90%) and Test (10%)...")
train_test_split_ratio = 0.10
seed = 42  # for reproducibility

# Using datasets built-in method
split_dataset = labeled_dataset.train_test_split(
    test_size = train_test_split_ratio, seed = seed, shuffle = True,
)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print("\nTrain/Test Split Complete.")

In [ ]:
# Let's define a function that performs chat messages formatting
# We will use "apply_chat_template()" which wraps the messages in a format the model understands
# Just like giving it a movie script so it knows who’s speaking and when to reply.
# It takes a list of messages (like a chat between a user and an assistant) and turns it into one long, properly formatted text, exactly how the model expects to read it.

def format_for_sft_gemma(example, tokenizer):

    # Define the conversation structure
    system_prompt = "Classify the sentiment of the following sentence from News as positive, negative, or neutral."
    user_prompt = f"Sentence: {example['text']}"
    assistant_response = example['sentiment'] # The target label

    messages = [
        {"role": "user", "content": f"{system_prompt}\n{user_prompt}"}, # Combine system/user for simplicity here
        {"role": "assistant", "content": assistant_response}
    ]
    # Apply the tokenizer's chat template.
    # tokenize = False: means we want the text output, not token IDs.
    # add_generation_prompt = False: means we’re NOT adding the assistant prompt to generate the response. We already provided the assistant's message
    formatted_text = tokenizer.apply_chat_template(messages, tokenize = False, add_generation_prompt = False)
    return {"text": formatted_text}

print("\nSupervised Fine Tuning (SFT) Trainer formatting function defined")

In [ ]:
# Let's test out the function!
# This code is setting up a Gemma language model (specifically google/gemma-3-1b-it) with 4-bit quantization using BitsAndBytesConfig
# for efficient loading which is useful on limited hardware like a laptop or free Colab GPU.

from tqdm.notebook import tqdm  # Progress bar meaning takadom "progress" in Arabic

# Let's choose the Gemma 3.1B Instruct model by Google
# It's a small instruction-tuned LLM that's good for chat, Q&A, etc.
os_model_id = "google/gemma-3-1b-it"

# Let's define the quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True, # Use 4-bit weights (saves lots of RAM)
    bnb_4bit_quant_type = "nf4", # nf4 is a better 4-bit format (Non-Float 4)
    bnb_4bit_compute_dtype = torch.float16 # Math is done in float16 for speed
)

# Let's load the Tokenizer associated with the model. Tokenizers convert text to tokens (numbers).
# It’s like the model’s language interpreter.
base_os_tokenizer = AutoTokenizer.from_pretrained(os_model_id)

# Set pad token if missing (Gemma often doesn't have one)
# A pad token is a special token used to make all input sequences the same length in a batch.
if base_os_tokenizer.pad_token is None:
    base_os_tokenizer.pad_token = base_os_tokenizer.eos_token
    print(f"Set pad_token to eos_token ({base_os_tokenizer.eos_token})")

In [ ]:
# Sample input dictionary
# <bos>                      → Beginning of sequence (tells the model: "Start reading")
# <start_of_turn>user        → Start of the user's turn
# Classify the sentiment...  → Instruction and user input
# <end_of_turn>              → End of user input
# <start_of_turn>model       → Start of the assistant/model's reply
# positive                   → The expected response
# <end_of_turn>              → End of assistant reply

example = {
    "text": "The economy is showing signs of recovery after a tough year.",
    "sentiment": "positive"
}

# Apply your formatting function
formatted_example = format_for_sft_gemma(example, base_os_tokenizer)

# Print the result
print("\n--- Sample Formatted Prompt ---")
print(formatted_example["text"])

In [ ]:
!pip install bitsandbytes

In [ ]:
# Load Base Model (Quantized)
quantization_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4", # Recommended type
    bnb_4bit_compute_dtype = torch.float16
)

base_os_tokenizer = AutoTokenizer.from_pretrained(os_model_id)

# Set pad token if missing (Gemma often doesn't have one)
if base_os_tokenizer.pad_token is None:
    base_os_tokenizer.pad_token = base_os_tokenizer.eos_token
    print(f"Set pad_token to eos_token ({base_os_tokenizer.eos_token})")

# Load model
base_os_model = AutoModelForCausalLM.from_pretrained(
    os_model_id,
    quantization_config = quantization_config,
    torch_dtype = torch.float16,
    device_map = "auto",
)
# Ensure model pad token ID is updated if tokenizer's was
base_os_model.config.pad_token_id = base_os_tokenizer.pad_token_id
print("Base Gemma model and tokenizer loaded successfully.")

In [ ]:
# Let's apply the SFT formatting function to our training and testing datasets
print("\nFormatting data for SFTTrainer (Gemma format)...")
sft_train_dataset = train_dataset.map(
              format_for_sft_gemma,
              fn_kwargs={"tokenizer": base_os_tokenizer},
              remove_columns=list(train_dataset.features)
              )
sft_test_dataset = test_dataset.map(
              format_for_sft_gemma,
              fn_kwargs={"tokenizer": base_os_tokenizer},
              remove_columns=list(test_dataset.features)
              )
print("SFTTrainer (Gemma) formatting complete.")
print("Sample SFT Gemma format:")
print(sft_train_dataset[0]['text'])

In [ ]:
# Perform Zero-Shot Classification Prompt Function (Gemma)
def create_zeroshot_prompt_gemma(sentence, tokenizer):

    """Creates zero-shot prompt using Gemma chat template."""
    system_prompt = f"Classify the sentiment of the following sentence from Financial News. Respond with ONLY ONE of the following labels: {', '.join(labels)}."
    user_prompt = f"Sentence: {sentence}"
    messages = [
        {"role": "user", "content": f"{system_prompt}\n{user_prompt}"},
    ]
    # Apply template, add_generation_prompt=True adds the assistant turn marker
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

In [ ]:
# Inference Function for Zero-Shot (Gemma)
# This function runs inference using the prompt generated above and returns the predicted sentiment label.
# It Gets the formatted prompt using the first function "create_zeroshot_prompt_gemma".
# It then tokenizes the prompt and sends it to the appropriate device (e.g., GPU):
def classify_zero_shot_os_gemma(sentence, model, tokenizer):

    prompt = create_zeroshot_prompt_gemma(sentence, tokenizer)
    inputs = tokenizer(prompt, return_tensors = "pt", truncation = True, max_length = 512).to(model.device)

    eos_id = tokenizer.eos_token_id          # eos_token_id: end of sentence
    pad_id = tokenizer.pad_token_id          # pad_token_id: padding token

    # Run the model to generate output, we are not performing gradient tracking since it's only inference!
    with torch.no_grad():

      outputs = model.generate(
          **inputs,
          max_new_tokens = 10, # Limits the number of tokens the model can generate
          eos_token_id = eos_id,
          pad_token_id = eos_id,
          do_sample = False
      )
    # Extracts and decode the generated text
    response_ids = outputs[0][inputs['input_ids'].shape[1]:]
    response_text = tokenizer.decode(response_ids, skip_special_tokens=True).strip()

    # print(f"Generated Text (Gemma): {response_text}")

    # Clean and Validate Output
    labels = ["neutral", "negative", "positive"]

    predicted_label = "Unknown"
    for label_text in labels:
        if label_text.lower() in response_text.lower(): # Simple check if label is present
              predicted_label = label_text
              break

    print(f"Predicted Label: {predicted_label}")
    return predicted_label

In [ ]:
true_labels = []

# Let's extract the ground truth (target output) which represents the True class
true_labels = [ex['sentiment'] for ex in test_dataset]
true_labels

In [ ]:
# --- Evaluate on Test Set ---
base_os_predictions = [] # Renaming for clarity if needed, but reusing is fine

# Let's perform inference on all testing datasets using the pre-trained Gemma LLM
for example in tqdm(test_dataset):
    predicted_label = classify_zero_shot_os_gemma(example['text'], base_os_model, base_os_tokenizer)
    base_os_predictions.append(predicted_label)


In [ ]:
print("\n--- Base Gemma Model: Zero-Shot Evaluation Results ---")
valid_indices = [i for i, p in enumerate(base_os_predictions) if p not in ["Error", "Unknown"]]
filtered_preds = [base_os_predictions[i] for i in valid_indices]
filtered_true = [true_labels[i] for i in valid_indices]

accuracy = accuracy_score(filtered_true, filtered_preds)
report = classification_report(filtered_true, filtered_preds, labels=labels, zero_division=0, target_names=labels)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)
# (Optional: show sample predictions)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

# Compute confusion matrix
cm = confusion_matrix(filtered_true, filtered_preds, labels=labels)

# Create heatmap
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels, cbar=False)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Base Gemma Model (Zero-Shot)")
plt.tight_layout()
plt.show()

In [ ]:

print("\n--- Preparing Model and Config for PEFT/LoRA Fine-tuning ---")

# --- Prepare model for k-bit training (important for quantized models) ---
base_os_model.gradient_checkpointing_enable()  # Saves memory during training
prepared_model = prepare_model_for_kbit_training(base_os_model)
print("Model prepared for k-bit training.")

print(prepared_model)


# --- LoRA Configuration ---
# Target modules often include query/key/value layers in attention blocks
# This depends on the model architecture (use print(prepared_model) to inspect layers)
# For Qwen-based models, common targets might be 'q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'
# Let's start with a reasonable default set
lora_config = LoraConfig(
    r = 16,  # LoRA rank (dimension of adapter matrices). Higher rank = more parameters, potentially better fit but slower. 8, 16, 32 are common.
    lora_alpha = 32,  # Scaling factor for LoRA weights (often 2*r).
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],  # Modules to apply LoRA to.
    lora_dropout = 0.05,  # Dropout probability for LoRA layers.
    bias = "none",  # Usually set to 'none'.
    task_type = "CAUSAL_LM",  # Task type for sequence generation.
)
print("LoRA Config created.")

# --- Apply PEFT to the model ---
# This adds the small LoRA layers into the big model. During fine-tuning, only these are updated — saving a lot of compute and memory
peft_model = get_peft_model(prepared_model, lora_config)



# The percentage (often < 1%) is small because LoRA only introduces and trains the parameters within the small adapter layers (`q_proj`, `k_proj`, etc. in our config),
# It does not train the original billions of parameters in the base model (which remain frozen).
# Training far fewer parameters requires significantly less GPU memory (VRAM), making it feasible to fine-tune large models on hardware like Colab's T4 GPU.
# It's also much faster than full fine-tuning.
print("PEFT model created.")
peft_model.print_trainable_parameters()

In [ ]:
# --- Training Arguments ---
output_dir = "./sentiment_finetuned_adapter"

training_args = TrainingArguments(
    output_dir = output_dir,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 2,
    learning_rate = 2e-4,
    num_train_epochs = 1,
    logging_steps = 25,
    save_strategy = "epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    max_grad_norm = 0.3,
    warmup_steps = 0.03 * len(sft_train_dataset) // (4 * 2),
    lr_scheduler_type = "constant",
    report_to = "none",
)
print("Training Arguments set.")

# --- Initialize SFTTrainer ---
# We pass the 'prepared_model' (base) instead of 'peft_model'
# because we are also providing the 'peft_config'.
trainer = SFTTrainer(
    model = prepared_model,
    args = training_args,
    train_dataset = sft_train_dataset,
    peft_config = lora_config,
    processing_class = base_os_tokenizer
)
print("SFTTrainer initialized.")

# --- Start Fine-tuning ---
print("\n--- Starting Fine-tuning... ---")
try:
    training_results = trainer.train()
    print("--- Fine-tuning Complete! ---")
    print(training_results)

    print(f"Saving LoRA adapter model to {output_dir}...")
    trainer.save_model(output_dir)
    base_os_tokenizer.save_pretrained(output_dir)
    print("Adapter and tokenizer saved.")

    del trainer
    torch.cuda.empty_cache()
    import gc
    gc.collect()
    print("Cleaned up training objects from memory.")
    fine_tuning_successful = True
except Exception as e:
    print(f"Error during fine-tuning: {e}")
    fine_tuning_successful = False

In [ ]:
tuned_os_predictions = []

adapter_path = "./sentiment_finetuned_adapter"

print(f"\n--- Loading Fine-tuned Open Source Model from {adapter_path} ---")

# --- Load Tokenizer (from saved adapter dir) ---
tuned_os_tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

# --- Load Base Model (Quantized) ---
# It's often cleaner to reload the base model before applying the adapter
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

base_model_reload = AutoModelForCausalLM.from_pretrained(
    os_model_id,  # The original base model ID
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)
# Set pad token ID on the reloaded model config if needed
if tuned_os_tokenizer.pad_token_id is not None:
    base_model_reload.config.pad_token_id = tuned_os_tokenizer.pad_token_id

# --- Load PEFT Adapter and Merge (or use directly) ---
# Load PeftModel directly (uses base + adapter without merging)
from peft import PeftModel

tuned_os_model = PeftModel.from_pretrained(base_model_reload, adapter_path)
print("Loaded PEFT model (base + adapter).")

# Ensure model is in evaluation mode
tuned_os_model.eval()
print("Fine-tuned OS model loaded successfully.")


# --- Evaluate on Test Set ---

# Assuming true_labels list was populated in previous steps
if "true_labels" not in locals() or not true_labels:
    print("Warning: true_labels not found from previous steps. Re-extracting.")
    true_labels = [ex["sentiment"] for ex in test_dataset]

for example in tqdm(test_dataset):  # Use tqdm for progress bar
    sentence = example["text"]
    # Use the SAME zero-shot function, but pass the TUNED model
    predicted_label = classify_zero_shot_os_gemma(sentence, tuned_os_model, tuned_os_tokenizer)
    tuned_os_predictions.append(predicted_label)

print("\n--- Fine-Tuned Open Source Model: Evaluation Results ---")
valid_indices_tuned = [i for i, p in enumerate(tuned_os_predictions) if p not in ["Error", "Unknown"]]
filtered_preds_tuned = [tuned_os_predictions[i] for i in valid_indices_tuned]
# Ensure we use the same true labels corresponding to the test set order
filtered_true_tuned = [
    true_labels[i] for i in valid_indices_tuned
]  # Use true_labels from Section 3 evaluation run


accuracy_tuned = accuracy_score(filtered_true_tuned, filtered_preds_tuned)
report_tuned = classification_report(
    filtered_true_tuned, filtered_preds_tuned, labels=labels, zero_division=0
)

print(f"Accuracy: {accuracy_tuned:.4f}")
print("\nClassification Report:")
print(report_tuned)

# --- Compare with Base OS Model ---
print("\n--- Comparison with Base OS Model ---")

print(f"Base OS Accuracy:      {accuracy:.4f}")
print(f"Fine-Tuned OS Accuracy: {accuracy_tuned:.4f}")
improvement = accuracy_tuned - accuracy
print(f"Improvement:           {improvement:+.4f}")


